# GDELT MCP Server — Capability Testing

This notebook exercises all capabilities of the **`mcp-gdelt`** server, which wraps the [GDELT DOC 2.0 API](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/) and the [GDELT Cloud Media Events API](https://docs.gdeltcloud.com/api-reference/media-events/media-events/top-media-event-clusters).  
It is used to validate query construction, parameter handling, and response parsing implemented in `src/mcp_gdelt/`.

### What is GDELT?
The **Global Database of Events, Language, and Tone (GDELT)** monitors news media worldwide in 65+ languages and provides real-time and historical access to:
- **Article metadata** — titles, URLs, domains, languages, source countries, tone
- **Visual Knowledge Graph (VGKG)** — AI-tagged and OCR-extracted metadata from news images
- **Media Event Clusters** — CAMEO-coded geopolitical events aggregated from global news (GDELT Cloud, from Jan 2025)

### What `mcp-gdelt` provides
| Tool | Description |
|------|-------------|
| `search_articles`     | Search 65-language global news index with keyword, Boolean, and date filters |
| `search_images`       | Search GDELT Visual Knowledge Graph by AI tag, caption, or OCR content |
| `search_media_events` | Search GDELT Cloud top media event clusters (CAMEO events, requires API key) |

---
**Notebook sections**
1. Environment setup
2. Direct client import
3. Article search — basic
4. Article search — Boolean operators
5. Article search — date ranges
6. Article search — sort orders
7. Image search — AI visual tags (`imagetag`)
8. Image search — web/caption tags (`imagewebtag`)
9. Image search — OCR metadata (`imageocrmeta`)
10. Multi-query comparison
11. Data analysis & visualisation
12. Media event clusters — basic search
13. Media event clusters — semantic search
14. Media event clusters — filters & analysis
15. Cleanup

## 1 · Environment Setup
Clone the repository and install `mcp-gdelt` in editable mode so we can import the internal async client directly.

In [ ]:
import os, subprocess, sys

REPO = "/content/mcp-servers"

# Clone once; skip if the directory already exists
if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/dshipley71/mcp-servers.git", REPO],
        check=True,
    )
    print(f"✓ Repository cloned → {REPO}")
else:
    print(f"✓ Repository already present at {REPO}")

# Install mcp-gdelt and all its dependencies (httpx, pydantic, …) in editable mode.
# -q suppresses the noisy pip output; remove -q to see full install log.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}/mcp-gdelt"],
    check=True,
)
print("✓ mcp-gdelt installed")

In [ ]:
# ------------------------------------------------------------------
# Optional: GDELT Cloud API key
# ------------------------------------------------------------------
# A GDELT Cloud API key (format: gdelt_sk_...) removes the strict
# anonymous rate limit and enables higher-throughput access.
# Obtain one at: https://docs.gdeltcloud.com/developers/api-keys
#
# The key is read from Colab Secrets (recommended) so it is never
# hard-coded in the notebook.  To add it:
#   1. Click the key icon in the left sidebar → "Secrets"
#   2. Add a secret named  GDELT_API_KEY  with your key as the value
#   3. Enable notebook access for that secret
# ------------------------------------------------------------------
import os

try:
    from google.colab import userdata
    GDELT_API_KEY = userdata.get("GDELT_API_KEY") or ""
except Exception:
    GDELT_API_KEY = os.environ.get("GDELT_API_KEY", "")

if GDELT_API_KEY:
    os.environ["GDELT_API_KEY"] = GDELT_API_KEY
    print("✓ API key loaded — authenticated access (rate limit: ~1 req/s)")
else:
    os.environ.pop("GDELT_API_KEY", None)
    print("ℹ No API key — anonymous access (rate limit: 1 req / 6 s)")

In [ ]:
# Standard library / third-party imports used throughout the notebook
import asyncio
import json
import os
from datetime import datetime, timedelta, timezone
from collections import Counter

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML, Image as IPImage

# mcp-gdelt internals
import sys
sys.path.insert(0, "/content/mcp-servers/mcp-gdelt/src")

from mcp_gdelt.config import config
from mcp_gdelt.services.gdelt_client import GDELTClient
from mcp_gdelt.types import (
    SearchArticlesInput,
    SearchImagesInput,
    SearchMediaEventsInput,
    GDELTArticle,
    GDELTImage,
    GDELTResolvedCluster,
)

print("✓ Imports complete")

In [ ]:
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
# Rate limiting, retry, and response caching are handled inside
# GDELTClient itself (see src/mcp_gdelt/services/gdelt_client.py).
# These wrappers simply unwrap the response into plain Python lists.
#
# Cache TTL is controlled by GDELT_CACHE_TTL (default 300 s).
# Set os.environ["GDELT_CACHE_TTL"] = "0" before importing to disable.

client = GDELTClient()

async def search_articles(**kwargs) -> list[GDELTArticle]:
    inp = SearchArticlesInput(**kwargs)
    resp = await client.search_articles(inp)
    return resp.articles or []

async def search_images(**kwargs) -> list[GDELTImage]:
    inp = SearchImagesInput(**kwargs)
    resp = await client.search_images(inp)
    return resp.images or []

def articles_to_df(articles: list[GDELTArticle]) -> pd.DataFrame:
    return pd.DataFrame([a.model_dump() for a in articles])

def images_to_df(images: list[GDELTImage]) -> pd.DataFrame:
    return pd.DataFrame([i.model_dump() for i in images])

def show_articles(articles: list[GDELTArticle], n: int = 10) -> None:
    """Pretty-print the first n articles."""
    df = articles_to_df(articles)[["seendate", "domain", "language", "sourcecountry", "title"]]
    display(df.head(n).style.set_properties(**{"text-align": "left"}))

print(f"✓ Helper functions ready  |  cache TTL: {config.gdelt_cache_ttl:.0f} s"
      f"  |  rate limit interval: {config.gdelt_rate_limit_interval:.1f} s")

In [ ]:
# Cache management utilities — run any time during the session.

def cache_stats() -> None:
    """Print live/expired/total cache entry counts."""
    s = client.cache_stats()
    print(f"Cache: {s['live']} live  |  {s['expired']} expired  |  {s['total']} total"
          f"  (TTL {config.gdelt_cache_ttl:.0f} s)")

def cache_clear() -> None:
    """Evict all cached responses (forces fresh API calls on next query)."""
    n = client.cache_clear()
    print(f"Cache cleared — {n} entries removed")

cache_stats()

In [ ]:
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
# Rate limiting and retry are handled inside GDELTClient itself.
# These wrappers simply unwrap the response into plain Python lists.

client = GDELTClient()

async def search_articles(**kwargs) -> list[GDELTArticle]:
    inp = SearchArticlesInput(**kwargs)
    resp = await client.search_articles(inp)
    return resp.articles or []

async def search_images(**kwargs) -> list[GDELTImage]:
    inp = SearchImagesInput(**kwargs)
    resp = await client.search_images(inp)
    return resp.images or []

def articles_to_df(articles: list[GDELTArticle]) -> pd.DataFrame:
    return pd.DataFrame([a.model_dump() for a in articles])

def images_to_df(images: list[GDELTImage]) -> pd.DataFrame:
    return pd.DataFrame([i.model_dump() for i in images])

def show_articles(articles: list[GDELTArticle], n: int = 10) -> None:
    """Pretty-print the first n articles."""
    df = articles_to_df(articles)[["seendate", "domain", "language", "sourcecountry", "title"]]
    display(df.head(n).style.set_properties(**{"text-align": "left"}))

print("✓ Helper functions ready")

---
## 3 · Article Search — Basic
Demonstrate a simple single-keyword search with default parameters.

In [ ]:
articles_basic = await search_articles(
    query="climate change",
    max_records=20,
    timespan="7d",
)

print(f"Results returned: {len(articles_basic)}")
show_articles(articles_basic)

In [ ]:
# Inspect a single article object
if articles_basic:
    first = articles_basic[0]
    print(json.dumps(first.model_dump(), indent=2))

---
## 4 · Article Search — Boolean Operators
GDELT supports `AND`, `OR`, and exact phrase matching with double quotes.

In [ ]:
# Exact phrase search
articles_phrase = await search_articles(
    query='"artificial intelligence" AND regulation',
    max_records=25,
    timespan="14d",
)
print(f'Exact phrase "artificial intelligence" AND regulation: {len(articles_phrase)} results')
show_articles(articles_phrase)

In [ ]:
# OR operator — broaden the search across two topics
articles_or = await search_articles(
    query='"solar energy" OR "wind energy"',
    max_records=30,
    timespan="7d",
)
print(f'"solar energy" OR "wind energy": {len(articles_or)} results')
show_articles(articles_or)

In [ ]:
# Multi-term AND — narrow the search
articles_and = await search_articles(
    query='"electric vehicles" AND "battery" AND "charging"',
    max_records=25,
    timespan="14d",
)
print(f'EV + battery + charging: {len(articles_and)} results')
show_articles(articles_and)

---
## 5 · Article Search — Date Ranges
Use `start_date_time` / `end_date_time` (format: `YYYYMMDDHHMMSS`) for precise windows,  
or the `timespan` shorthand (`15min`, `1h`, `24h`, `7d`, `1month`, …).

In [ ]:
# Timespan shorthand — last 24 hours
articles_24h = await search_articles(
    query="earthquake",
    max_records=20,
    timespan="24h",
)
print(f"Earthquake news — last 24 h: {len(articles_24h)} results")
show_articles(articles_24h)

In [ ]:
# Absolute date range — compute a 3-day window ending now
now = datetime.now(timezone.utc)
three_days_ago = now - timedelta(days=3)

start_dt = three_days_ago.strftime("%Y%m%d%H%M%S")
end_dt   = now.strftime("%Y%m%d%H%M%S")

print(f"Window: {start_dt} → {end_dt}")

articles_range = await search_articles(
    query="cybersecurity breach",
    max_records=20,
    start_date_time=start_dt,
    end_date_time=end_dt,
)
print(f"Cybersecurity breach — 3-day window: {len(articles_range)} results")
show_articles(articles_range)

In [ ]:
# All available timespan shorthand examples
timespans = ["15min", "1h", "4h", "12h", "24h", "3d", "7d", "2weeks", "1month"]
counts = {}

for ts in timespans:
    try:
        articles = await search_articles(query="economy", max_records=1, timespan=ts)
        # max_records=1 just to test the timespan param; GDELT may return 0–1
        counts[ts] = len(articles)
        print(f"  timespan={ts:10s} → {counts[ts]} result(s) (limited to 1)")
    except Exception as exc:
        counts[ts] = -1
        print(f"  timespan={ts:10s} → ERROR: {exc}")

---
## 6 · Article Search — Sort Orders
GDELT supports five sort modes: `DateDesc`, `DateAsc`, `ToneAsc`, `ToneDesc`, `HybridRel`.

In [ ]:
sort_modes = ["DateDesc", "DateAsc", "ToneAsc", "ToneDesc", "HybridRel"]

for mode in sort_modes:
    arts = await search_articles(
        query="global health",
        max_records=5,
        timespan="7d",
        sort=mode,
    )
    print(f"\n--- sort={mode} ({len(arts)} results) ---")
    for a in arts:
        print(f"  [{a.seendate}] {a.domain}: {a.title[:80]}")

---
## 7 · Image Search — AI Visual Tags (`imagetag`)
`imagetag` uses GDELT's computer-vision pipeline to identify visual content in images.

In [ ]:
images_fire = await search_images(
    query="fire",
    max_records=20,
    timespan="7d",
    image_type="imagetag",
)
print(f"Images tagged 'fire' (AI visual): {len(images_fire)}")
display(images_to_df(images_fire).head(10))

In [ ]:
# Display the first few images inline
for img in images_fire[:4]:
    try:
        resp = httpx.get(img.url, timeout=10, follow_redirects=True)
        if resp.status_code == 200:
            display(IPImage(data=resp.content, width=300))
            print(f"  {img.url}  ({img.width}×{img.height}, {img.format})")
    except Exception as exc:
        print(f"  Could not load {img.url}: {exc}")

In [ ]:
# Try several visual tags
visual_tags = ["protest", "flood", "military", "summit", "hospital"]

tag_counts = {}
for tag in visual_tags:
    imgs = await search_images(query=tag, max_records=50, timespan="7d", image_type="imagetag")
    tag_counts[tag] = len(imgs)
    print(f"  imagetag='{tag}': {len(imgs)} images")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tag_counts.keys(), tag_counts.values(), color="steelblue")
ax.set_title("Image count by AI visual tag (last 7 days)")
ax.set_ylabel("# images")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 8 · Image Search — Web/Caption Tags (`imagewebtag`)
`imagewebtag` searches text extracted from captions, alt text, and surrounding context.

In [ ]:
images_web = await search_images(
    query="climate summit",
    max_records=20,
    timespan="1month",
    image_type="imagewebtag",
)
print(f"imagewebtag 'climate summit': {len(images_web)} results")
display(images_to_df(images_web).head(10))

In [ ]:
# Compare imagetag vs imagewebtag for the same query
query_compare = "earthquake"

imgs_tag = await search_images(query=query_compare, max_records=50, timespan="7d", image_type="imagetag")
imgs_web = await search_images(query=query_compare, max_records=50, timespan="7d", image_type="imagewebtag")

print(f"Query: '{query_compare}'")
print(f"  imagetag    → {len(imgs_tag)} images")
print(f"  imagewebtag → {len(imgs_web)} images")

---
## 9 · Image Search — OCR Metadata (`imageocrmeta`)
`imageocrmeta` finds images whose OCR-extracted text matches the query.

In [ ]:
images_ocr = await search_images(
    query="press conference",
    max_records=20,
    timespan="14d",
    image_type="imageocrmeta",
)
print(f"imageocrmeta 'press conference': {len(images_ocr)} results")
display(images_to_df(images_ocr).head(10))

In [ ]:
# All three image types side-by-side for one query
query_all = "parliament"
image_types = ["imagetag", "imagewebtag", "imageocrmeta"]

type_counts = {}
for itype in image_types:
    imgs = await search_images(query=query_all, max_records=50, timespan="14d", image_type=itype)
    type_counts[itype] = len(imgs)
    print(f"  {itype:15s} → {len(imgs)} images")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(type_counts.keys(), type_counts.values(), color=["#2196F3", "#4CAF50", "#FF9800"])
ax.set_title(f"Image search types for '{query_all}' (last 14 days)")
ax.set_ylabel("# images")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 10 · Multi-Query Comparison
Run several queries in parallel (using `asyncio.gather`) and compare results.

In [ ]:
topics = [
    "nuclear energy",
    "renewable energy",
    "fossil fuels",
    "hydrogen fuel",
    "carbon capture",
]

# Run sequentially — the rate limiter in search_articles enforces the
# 5 s GDELT gap, so concurrent gather() would just queue up anyway.
counts_by_topic = {}
for t in topics:
    arts = await search_articles(query=t, max_records=50, timespan="7d")
    counts_by_topic[t] = len(arts)
    print(f"  {t:25s}: {len(arts)}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(counts_by_topic.keys()), list(counts_by_topic.values()), color="teal")
ax.set_xlabel("# articles (last 7 days, max 50)")
ax.set_title("Energy topic coverage in GDELT (last 7 days)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 11 · Data Analysis & Visualisation
Work with the article metadata to analyse language coverage, source countries, and domain diversity.

In [ ]:
# Fetch a larger result set for analysis
ANALYSIS_QUERY = "technology"
ANALYSIS_MAX   = 250

articles_analysis = await search_articles(
    query=ANALYSIS_QUERY,
    max_records=ANALYSIS_MAX,
    timespan="7d",
    sort="DateDesc",
)
df = articles_to_df(articles_analysis)
print(f"Fetched {len(df)} articles for '{ANALYSIS_QUERY}' (last 7 days)")
df.head()

In [ ]:
# ── Language distribution ──────────────────────────────────────────
lang_counts = df["language"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 4))
lang_counts.plot(kind="bar", ax=ax, color="slateblue")
ax.set_title(f"Top languages — '{ANALYSIS_QUERY}' coverage (last 7 days)")
ax.set_xlabel("Language")
ax.set_ylabel("# articles")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Source country distribution ───────────────────────────────────
country_counts = df["sourcecountry"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 4))
country_counts.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title(f"Top source countries — '{ANALYSIS_QUERY}' (last 7 days)")
ax.set_xlabel("Country")
ax.set_ylabel("# articles")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Top domains ───────────────────────────────────────────────────
domain_counts = df["domain"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
domain_counts.plot(kind="barh", ax=ax, color="mediumseagreen")
ax.set_title(f"Top 20 domains — '{ANALYSIS_QUERY}' (last 7 days)")
ax.set_xlabel("# articles")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Volume over time (hourly bins) ───────────────────────────────
if "seendate" in df.columns and df["seendate"].notna().any():
    df["seendate_parsed"] = pd.to_datetime(df["seendate"], format="%Y%m%dT%H%M%SZ", errors="coerce")
    hourly = df.set_index("seendate_parsed").resample("6h").size()

    fig, ax = plt.subplots(figsize=(12, 4))
    hourly.plot(ax=ax, color="royalblue", marker="o", markersize=4)
    ax.set_title(f"Article volume over time — '{ANALYSIS_QUERY}' (6-hour bins)")
    ax.set_ylabel("# articles")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
else:
    print("No parseable dates in result set")

In [ ]:
# ── Summary statistics ───────────────────────────────────────────
print("=" * 55)
print(f" GDELT Query Summary — '{ANALYSIS_QUERY}'")
print("=" * 55)
print(f" Total articles     : {len(df)}")
print(f" Unique domains     : {df['domain'].nunique()}")
print(f" Unique languages   : {df['language'].nunique()}")
print(f" Unique countries   : {df['sourcecountry'].nunique()}")
if "seendate_parsed" in df.columns:
    valid_dates = df["seendate_parsed"].dropna()
    if not valid_dates.empty:
        print(f" Earliest article   : {valid_dates.min()}")
        print(f" Latest article     : {valid_dates.max()}")
print("=" * 55)

---
## 12 · Media Event Clusters — Basic Search

The **GDELT Cloud Media Events API** aggregates global news into CAMEO-coded event clusters.  
Each cluster groups related articles, resolves CAMEO actor/event codes, Goldstein scale, tone,  
and links key entities (people, organisations) to Wikipedia.

> **Requires a GDELT Cloud API key** (format: `gdelt_sk_...`).  
> Set it via Colab Secrets → `GDELT_API_KEY` (see cell 2 above).  
> Obtain a key at: https://docs.gdeltcloud.com/developers/api-keys
>
> Data starts from **January 2025**, updated hourly.

In [ ]:
# ------------------------------------------------------------------
# Helper: search_media_events wrapper
# ------------------------------------------------------------------
async def search_media_events(**kwargs) -> list[GDELTResolvedCluster]:
    inp = SearchMediaEventsInput(**kwargs)
    resp = await client.search_media_events(inp)
    return resp.clusters or []

def show_clusters(clusters: list[GDELTResolvedCluster], detail: bool = False) -> None:
    """Print a compact summary of each cluster."""
    for i, c in enumerate(clusters, 1):
        rm = c.resolved_metrics or None
        loc = rm.primary_location if rm else None
        event = rm.primary_event if rm else None
        loc_str = ", ".join(filter(None, [
            loc.adm1_name if loc else None,
            loc.country_name if loc else None,
        ])) if loc else "—"
        print(
            f"{i:2d}. [{c.category or '?':22s}] [{c.scope or '?':8s}]  "
            f"articles={c.article_count or 0:4d}  "
            f"goldstein={rm.avg_goldstein:+.1f}" if rm and rm.avg_goldstein is not None else
            f"{i:2d}. [{c.category or '?':22s}] [{c.scope or '?':8s}]  "
            f"articles={c.article_count or 0:4d}"
        )
        print(f"    {c.cluster_label or '(no label)'}")
        if loc_str != "—":
            print(f"    Location : {loc_str}")
        if event and event.description:
            print(f"    Event    : {event.description}")
        if detail and c.linked_entities:
            entities = [e.name for e in c.linked_entities[:5] if e.name]
            if entities:
                print(f"    Entities : {', '.join(entities)}")
        print()

if config.gdelt_api_key:
    print("✓ GDELT Cloud API key present — search_media_events is available")
else:
    print("⚠ No GDELT Cloud API key.  Set GDELT_API_KEY via Colab Secrets to use search_media_events.")

In [ ]:
# Basic fetch — top 10 global clusters from the last 7 days
# detail="full" includes CAMEO metrics, representative articles, and entity links.
clusters_basic = await search_media_events(
    days=7,
    limit=10,
    detail="full",
)
print(f"Clusters returned: {len(clusters_basic)}\n")
show_clusters(clusters_basic, detail=True)

In [ ]:
# Inspect one cluster in full detail
if clusters_basic:
    print(json.dumps(clusters_basic[0].model_dump(exclude_none=True), indent=2))

---
## 13 · Media Event Clusters — Semantic Search

The `search` parameter uses text-embedding similarity (HNSW index) to rank clusters by relevance  
to a natural-language query rather than by article count.  
It can be combined with any other filter.

In [ ]:
# Semantic search — conflict events involving climate / environmental policy
clusters_climate = await search_media_events(
    search="climate policy international agreement",
    days=7,
    limit=10,
    detail="standard",
)
print(f"Semantic search 'climate policy international agreement': {len(clusters_climate)} clusters\n")
show_clusters(clusters_climate)

In [ ]:
# Semantic search combined with scope + category filter
clusters_conflict = await search_media_events(
    search="military attack armed conflict",
    days=7,
    limit=10,
    scope="global",
    category="conflict_security",
    detail="standard",
)
print(f"Semantic + conflict_security + global scope: {len(clusters_conflict)} clusters\n")
show_clusters(clusters_conflict)

---
## 14 · Media Event Clusters — Filters & Analysis

Demonstrate the full set of structured filters: country, actor, event type, Goldstein scale,  
tone range, quad class, and pagination.  Then build a DataFrame for analysis.

In [ ]:
# ── Country filter — events in or about France ────────────────────
clusters_france = await search_media_events(
    country="France",
    days=7,
    limit=10,
    detail="standard",
)
print(f"Country='France': {len(clusters_france)} clusters\n")
show_clusters(clusters_france)

In [ ]:
# ── Goldstein scale filter — strongly negative events (conflict) ──
# Goldstein scale: -10 (most conflictual) to +10 (most cooperative)
clusters_negative = await search_media_events(
    days=7,
    limit=10,
    goldstein_min=-10.0,
    goldstein_max=-5.0,
    detail="standard",
)
print(f"Goldstein -10 to -5 (high conflict): {len(clusters_negative)} clusters\n")
show_clusters(clusters_negative)

In [ ]:
# ── Quad class filter — Material Conflict (4) ────────────────────
# Quad class: 1=Verbal Cooperation, 2=Material Cooperation,
#             3=Verbal Conflict,    4=Material Conflict
clusters_qc4 = await search_media_events(
    days=7,
    limit=10,
    quad_class=4,
    detail="standard",
)
print(f"Quad class 4 (Material Conflict): {len(clusters_qc4)} clusters\n")
show_clusters(clusters_qc4)

# ── Pagination demo — page 1 and page 2 ──────────────────────────
clusters_p1 = await search_media_events(days=7, limit=5, offset=0,  detail="summary")
clusters_p2 = await search_media_events(days=7, limit=5, offset=5,  detail="summary")
print(f"\nPage 1 (offset=0):  {[c.cluster_label[:50] if c.cluster_label else '?' for c in clusters_p1]}")
print(f"Page 2 (offset=5):  {[c.cluster_label[:50] if c.cluster_label else '?' for c in clusters_p2]}")

---
## 15 · Cleanup
Close the async HTTP client when the notebook session is done.

---
## 12 · Cleanup
Close the async HTTP client when the notebook session is done.

---
## Quick Reference

### `search_articles` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Keywords, `"exact phrase"`, `AND` / `OR` |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | `15min` · `1h` · `24h` · `7d` · `1month` |
| `sort` | str | `DateDesc` | `DateDesc` · `DateAsc` · `ToneAsc` · `ToneDesc` · `HybridRel` |
| `start_date_time` | str | — | `YYYYMMDDHHMMSS` |
| `end_date_time` | str | — | `YYYYMMDDHHMMSS` |

### `search_images` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Visual concept, caption text, or OCR text |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | Same shorthands as above |
| `image_type` | str | `imagetag` | `imagetag` · `imagewebtag` · `imageocrmeta` |

### `search_media_events` parameters (requires API key)
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `days` | int | 1 | Window size in days (1–30) |
| `date` | str | today UTC | Anchor end date (`YYYY-MM-DD`) |
| `limit` | int | 10 | Clusters to return (1–50) |
| `offset` | int | 0 | Pagination skip |
| `detail` | str | `full` | `summary` · `standard` · `full` |
| `search` | str | — | Natural-language semantic search |
| `category` | str | — | `conflict_security`, `politics_governance`, `crime_justice`, `economy_business`, `science_health`, `disaster_emergency`, `society_culture`, `technology` |
| `scope` | str | — | `local` · `national` · `global` |
| `actor_country` | str | — | CAMEO ISO-3 code (e.g. `USA`) |
| `event_type` | str | — | CAMEO root code (e.g. `14`=Protest) |
| `country` | str | — | Full name, ISO-3, or FIPS-2 code |
| `goldstein_min` | float | — | Min Goldstein scale (−10 to +10) |
| `goldstein_max` | float | — | Max Goldstein scale |
| `tone_min` | float | — | Min average tone |
| `tone_max` | float | — | Max average tone |
| `quad_class` | int | — | 1=Verbal Coop · 2=Material Coop · 3=Verbal Conflict · 4=Material Conflict |
| `domain` | str | — | Filter by news domain (e.g. `reuters.com`) |
| `language` | str | — | Language name or ISO-639-1 code |

### API endpoints
| Tool | Endpoint |
|------|----------|
| `search_articles` / `search_images` | `https://api.gdeltproject.org/api/v2/doc/doc` |
| `search_media_events` | `https://gdeltcloud.com/api/v1/media-events` |

### Useful resources
- [GDELT DOC 2.0 API documentation](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/)
- [GDELT Cloud Media Events API reference](https://docs.gdeltcloud.com/api-reference/media-events/media-events/top-media-event-clusters)
- [GDELT Cloud API keys](https://docs.gdeltcloud.com/developers/api-keys)
- [GDELT Project homepage](https://www.gdeltproject.org/)
- [mcp-gdelt README](https://github.com/dshipley71/mcp-servers/tree/main/mcp-gdelt)

---
## Quick Reference

### `search_articles` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Keywords, `"exact phrase"`, `AND` / `OR` |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | `15min` · `1h` · `24h` · `7d` · `1month` |
| `sort` | str | `DateDesc` | `DateDesc` · `DateAsc` · `ToneAsc` · `ToneDesc` · `HybridRel` |
| `start_date_time` | str | — | `YYYYMMDDHHMMSS` |
| `end_date_time` | str | — | `YYYYMMDDHHMMSS` |

### `search_images` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Visual concept, caption text, or OCR text |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | Same shorthands as above |
| `image_type` | str | `imagetag` | `imagetag` · `imagewebtag` · `imageocrmeta` |

### GDELT DOC 2.0 API base URL
```
https://api.gdeltproject.org/api/v2/doc/doc
```

### Useful resources
- [GDELT DOC 2.0 API documentation](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/)
- [GDELT Project homepage](https://www.gdeltproject.org/)
- [mcp-gdelt README](https://github.com/dshipley71/mcp-servers/tree/main/mcp-gdelt)